# ArchNetv2-openings — CubiCasa5K training (Colab / Runpod)

Faithful reimplementation of:
> Xu, Jha, Mehadi, Mandal. *Multiscale object detection on complex architectural floor plans.* Automation in Construction 165 (2024) 105486.

Trains the 4-detection-head + AC-CBAM model on CubiCasa5K for two classes: **door** and **window**.

Expected GPU: T4 (paper) or anything bigger. Batch=4 fits on a T4 at 640×640.

## 1. Environment

In [ ]:
!pip install -q ultralytics==8.4.58 pillow
!nvidia-smi | head -20

## 2. Pull this repo
Replace the URL with your own fork if needed.

In [ ]:
import os, sys
REPO = '/content/floorplan-openings-detector'
if not os.path.isdir(REPO):
    !git clone https://github.com/miladmirzazadeh/floorplan-openings-detector.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)

## 3. Download CubiCasa5K (~9 GB)

The official mirror is on Zenodo. Adjust the URL to whatever is current.

In [ ]:
import os
os.makedirs('/content/cubicasa5k_raw', exist_ok=True)
%cd /content/cubicasa5k_raw
# Replace with the current Zenodo / Kaggle mirror URL for CubiCasa5K:
!wget -q --show-progress https://zenodo.org/record/2613548/files/cubicasa5k.zip
!unzip -q cubicasa5k.zip
!ls

## 4. Convert to YOLO format (door/window only)

In [ ]:
%cd {REPO}
!python -m archnetv2.scripts.prepare_cubicasa5k \
    --src /content/cubicasa5k_raw/cubicasa5k \
    --dst /content/cubicasa5k_yolo \
    --quality all

## 5. (Optional) 8× offline augmentation matching the paper

Skip this if you want to rely on Ultralytics' runtime augmentation only — 
CubiCasa5K is already ~5k plans, so 8× expansion may be overkill.

In [ ]:
# !python -m archnetv2.scripts.expand_dataset --root /content/cubicasa5k_yolo --splits train

## 6. Sanity-check the model build

In [ ]:
from archnetv2.model import build_archnetv2
yolo = build_archnetv2(nc=2, verbose=True)
print('Strides:', yolo.model.stride.tolist())
print('Params (M):', sum(p.numel() for p in yolo.model.parameters())/1e6)

## 7. Train (paper hyperparameters: epochs=500, batch=4, patience=100)

In [ ]:
!python -m archnetv2.train \
    --data /content/cubicasa5k_yolo/data.yaml \
    --epochs 500 \
    --batch 4 \
    --imgsz 640 \
    --patience 100 \
    --device 0 \
    --name archnetv2_openings_v1

## 8. Inspect best.pt and run inference on a test image

In [ ]:
import glob
!ls runs/archnetv2/archnetv2_openings_v1/weights/
# Pick any converted test image to sanity-check predictions.
test_img = sorted(glob.glob('/content/cubicasa5k_yolo/images/test/*.png'))[0]
print('Testing on:', test_img)
!python -m archnetv2.predict \
    --weights runs/archnetv2/archnetv2_openings_v1/weights/best.pt \
    --source {test_img} \
    --json /content/pred.json \
    --save-vis

## 9. Persist best.pt back to your machine

Easiest: download via the Files panel, or push to Google Drive.

In [ ]:
# from google.colab import files
# files.download('runs/archnetv2/archnetv2_openings_v1/weights/best.pt')